# Original YOLO12-Small pretrained — RDD2022 China (Multi-seed)

Baseline YOLO12s tanpa modifikasi arsitektur. Notebook ini memasang Ultralytics resmi, melakukan `train → val → test`, dan menyimpan seluruh hasil ke ZIP. Gunakan hyperparameter serta split data yang sama dengan eksperimen modifikasi agar perbandingan mAP adil.

Multi-seed: seed 42, 123, 456 dengan hyperparameter identik. Hasil per seed disimpan sebagai ZIP terpisah, lalu diringkas sebagai mean ± std.

In [ ]:
# 1. Install Ultralytics resmi. Aktifkan GPU dan Internet pada Kaggle.
import json
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.135'], check=True)

import torch
import ultralytics
# Versi dikunci sama dengan run seed 42 lama agar seed 123/456 setara.
assert ultralytics.__version__ == '8.4.135', f'Ultralytics {ultralytics.__version__} != 8.4.135; restart session lalu jalankan ulang.'
from ultralytics import YOLO

WORKDIR = Path('/kaggle/working')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Device      : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

# Protokol run seed 42 lama: PyTorch 2.10.0+cu128 dan Tesla T4. Perbedaan tidak menghentikan run,
# tetapi wajib dicatat di log sesi (README_SOP.md).
if torch.__version__ != '2.10.0+cu128' or not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    print('WARNING: environment berbeda dari run seed 42 lama — catat di log sesi README_SOP.md.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 642.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 2.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

ENVIRONMENT
Python      : 3.13.15
PyTorch     : 2.11.0+cu128
Ultralytics : 8.4.135
Device      : 0
GPU         : Tesla T4


In [ ]:
# 2. Dataset RDD2022 China dan hyperparameter baseline.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS = 0, 2
# Multi-seed: hyperparameter identik untuk setiap seed; hanya SEED yang berbeda.
SEEDS = [42, 123, 456]
# Batas sesi Kaggle 12 jam. Bila tiga seed tidak muat dalam satu sesi, jalankan sebagian,
# misalnya RUN_SEEDS = [42] lalu [123, 456] pada sesi berikutnya.
# Seed 42 memakai hasil run lama (lihat README_SOP.md); sesi ini menjalankan:
RUN_SEEDS = [123, 456]
# Opsional: folder hasil sesi sebelumnya (Kaggle input) untuk agregasi mean ± std.
# '/kaggle/input' dicari rekursif: hasil seed 42 lama dan output sesi sebelumnya otomatis ikut.
EXTRA_RESULT_DIRS = ['/kaggle/input']
EXPERIMENT_NAME = 'yolo12s_original_pretrained_ch_rdd2022'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
print(DATA_YAML.read_text(encoding='utf-8'))
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seeds={SEEDS}, run_seeds={RUN_SEEDS}')

# Pemeriksaan awal (sebelum training berjam-jam): seed yang sudah punya hasil vs seed sesi ini.
PREVIOUS_SEEDS = sorted({
    json.loads(p.read_text(encoding='utf-8'))['seed']
    for d in EXTRA_RESULT_DIRS if Path(d).is_dir()
    for p in Path(d).rglob(f'{EXPERIMENT_NAME}_seed*_evaluation_metrics.json')
})
print(f'Seed dengan hasil di input : {PREVIOUS_SEEDS}')
print(f'Seed dijalankan sesi ini   : {RUN_SEEDS}')
if 42 not in PREVIOUS_SEEDS and 42 not in RUN_SEEDS:
    raise FileNotFoundError('Hasil seed 42 lama tidak ditemukan. Add Input dataset seed42-previous (README_SOP.md langkah A).')
if set(PREVIOUS_SEEDS) & set(RUN_SEEDS):
    raise ValueError(f'Seed {sorted(set(PREVIOUS_SEEDS) & set(RUN_SEEDS))} sudah punya hasil; '
                     'hapus dari RUN_SEEDS agar kuota GPU tidak terbuang.')
pending = sorted(set(SEEDS) - set(PREVIOUS_SEEDS) - set(RUN_SEEDS))
if pending:
    print(f'Catatan: seed {pending} dijadwalkan untuk sesi berikutnya.')

path: /kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair

epochs=160, imgsz=640, batch=16, nbs=64, optimizer=SGD, lr0=0.01, seeds=[42, 123, 456], run_seeds=[123, 456]
Seed dengan hasil di input : [42]
Seed dijalankan sesi ini   : [123, 456]


In [ ]:
# 3. Satu run per seed: train → validation → test → ZIP (isi sama dengan notebook seed 42).
import gc, shutil, statistics

def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def seed_paths(seed):
    name = f'{EXPERIMENT_NAME}_seed{seed}'
    return (name, WORKDIR / f'{name}_evaluation_metrics.json',
            WORKDIR / f'{name}_config.json', WORKDIR / f'{name}_results.zip')

def write_seed_zip(zip_path, archive_items):
    tmp_path = zip_path.with_suffix('.zip.tmp')
    files = []
    for item in archive_items:
        if item is None or not item.exists():
            continue
        files.extend([item] if item.is_file() else [p for p in item.rglob('*') if p.is_file()])
    files = list(dict.fromkeys(files))
    with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for file_path in files:
            archive.write(file_path, file_path.relative_to(WORKDIR).as_posix())
    tmp_path.replace(zip_path)  # ZIP final = penanda seed selesai.
    return len(files)

def prepare_seed(seed):
    name, evaluation_json, _, zip_path = seed_paths(seed)
    if zip_path.is_file() and evaluation_json.is_file():
        log_section(f'SEED {seed} SUDAH SELESAI — DILEWATI')
        return json.loads(evaluation_json.read_text(encoding='utf-8'))
    # Hapus sisa run seed ini yang terputus agar artefak seed tidak tercampur.
    for suffix in ('', '_val', '_test', '_test_predictions'):
        shutil.rmtree(RUNS_DIR / f'{name}{suffix}', ignore_errors=True)
    return None

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

def run_seed(SEED):
    name, EVALUATION_JSON, RUN_CONFIG, ZIP_PATH = seed_paths(SEED)
    done = prepare_seed(SEED)
    if done is not None:
        return done

    # Training original pretrained YOLO12s. yolo12s.pt akan diunduh otomatis dari Ultralytics bila belum tersedia.
    log_section(f'TRAINING — ORIGINAL PRETRAINED YOLO12S — SEED {SEED}')
    model = YOLO('yolo12s.pt')
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
        device=DEVICE, workers=WORKERS,
        project=str(RUNS_DIR), name=name, exist_ok=True,
        pretrained=True, optimizer=OPTIMIZER,
        lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
        cos_lr=False, patience=PATIENCE, seed=SEED,
        plots=True, verbose=True,
    )
    RUN_DIR = Path(model.trainer.save_dir)
    BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
    print(f'Run directory: {RUN_DIR}')
    print(f'Best weights : {BEST_PT}')
    del model
    free_gpu_memory()

    # Validasi best.pt pada validation split.
    log_section(f'VALIDATION — BEST.PT — SEED {SEED}')
    best_model = YOLO(str(BEST_PT))
    val_metrics = best_model.val(
        data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        project=str(RUNS_DIR), name=f'{name}_val', exist_ok=True, plots=True,
    )
    VAL_REPORT = metric_summary(val_metrics)
    print(json.dumps(VAL_REPORT, indent=2))

    # Uji best.pt pada test split.
    log_section(f'TEST — BEST.PT — SEED {SEED}')
    test_label_dir = DATA_ROOT / 'test' / 'labels'
    if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
        test_metrics = best_model.val(
            data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
            project=str(RUNS_DIR), name=f'{name}_test', exist_ok=True, plots=True,
        )
        TEST_REPORT = metric_summary(test_metrics)
        TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
        del test_metrics
    else:
        predictions = best_model.predict(
            source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE, conf=0.25,
            save=True, save_txt=True, project=str(RUNS_DIR),
            name=f'{name}_test_predictions', exist_ok=True,
        )
        TEST_REPORT = {'status': 'test labels unavailable; prediction only'}
        TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR / f'{name}_test_predictions'
        del predictions
    VAL_DIR = Path(val_metrics.save_dir)
    del val_metrics, best_model
    free_gpu_memory()

    # Simpan metrik dan konfigurasi seed ini, lalu kompres artefak ke ZIP.
    EVALUATION_REPORT = {'seed': SEED, 'validation': VAL_REPORT, 'test': TEST_REPORT}
    EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
    RUN_CONFIG.write_text(json.dumps({
        'model': 'yolo12s.pt', 'dataset_root': str(DATA_ROOT), 'data_yaml': str(DATA_YAML),
        'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS,
        'optimizer': OPTIMIZER, 'lr0': LR0, 'momentum': MOMENTUM,
        'weight_decay': WEIGHT_DECAY, 'seed': SEED, 'seeds': SEEDS,
        'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
    }, indent=2), encoding='utf-8')
    count = write_seed_zip(ZIP_PATH, [
        RUN_DIR, VAL_DIR, TEST_OUTPUT_DIR, DATA_YAML, RUN_CONFIG, EVALUATION_JSON,
    ])
    log_section(f'SEED {SEED} COMPLETED')
    print(json.dumps(EVALUATION_REPORT, indent=2))
    print(f'ZIP created : {ZIP_PATH} ({count} files)')
    return EVALUATION_REPORT

In [ ]:
# 4. Jalankan semua seed secara berurutan; memori GPU dibersihkan setelah setiap seed.
ALL_REPORTS = {}
for seed in RUN_SEEDS:
    ALL_REPORTS[seed] = run_seed(seed)
    free_gpu_memory()
    if torch.cuda.is_available():
        print(f'GPU memory setelah seed {seed}: {torch.cuda.memory_allocated() / 1024**2:.0f} MiB allocated')


TRAINING — ORIGINAL PRETRAINED YOLO12S — SEED 123
New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ch_rdd2022.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=160, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/160      5.98G      1.672       2.15      1.575         28        640: 100% ━━━━━━━━━━━━ 219/219 2.4it/s 1:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.7it/s 5.2s
                   all        438        876      0.578      0.379      0.394      0.194

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/160      5.97G      1.831      2.319      1.691         47        640: 100% ━━━━━━━━━━━━ 219/219 2.4it/s 1:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 2.7it/s 5.2s
                   all        438        876      0.149      0.197     0.0916     0.0407

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/160      5.97G      1.925      2.427      1.788         59        640: 100% ━━━━━━━━━━━━ 219/219 2.5it/s 1:29
                 Class     Images  Instances      Box(P 

In [ ]:
# 5. Agregasi mean ± std seluruh seed yang sudah selesai, lalu ZIP ringkasan.
pattern = f'{EXPERIMENT_NAME}_seed*_evaluation_metrics.json'
found = {}
for path in sorted(WORKDIR.glob(pattern)) + [p for d in EXTRA_RESULT_DIRS for p in sorted(Path(d).rglob(pattern))]:
    report = json.loads(path.read_text(encoding='utf-8'))
    found.setdefault(report['seed'], (path, report))

def mean_std(values):
    return {'mean': statistics.mean(values), 'std': statistics.stdev(values) if len(values) > 1 else 0.0,
            'values': values}

METRIC_KEYS = ('precision', 'recall', 'map50', 'map50_95')
SUMMARY = {'experiment': EXPERIMENT_NAME, 'seeds': sorted(found)}
for split_key in ('validation', 'test'):
    # Split test tanpa label (prediction only) tidak memiliki metrik sehingga tidak diagregasi.
    reports = [found[s][1][split_key] for s in sorted(found) if 'map50' in found[s][1].get(split_key, {})]
    if reports:
        SUMMARY[split_key] = {k: mean_std([r[k] for r in reports]) for k in METRIC_KEYS}

SUMMARY_JSON = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.json'
SUMMARY_JSON.write_text(json.dumps(SUMMARY, indent=2), encoding='utf-8')
SUMMARY_CSV = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.csv'
rows = ['seed,split,' + ','.join(METRIC_KEYS)]
for s in sorted(found):
    for split_key in ('validation', 'test'):
        r = found[s][1].get(split_key, {})
        if 'map50' in r:
            rows.append(f'{s},{split_key},' + ','.join(f'{r[k]:.6f}' for k in METRIC_KEYS))
for split_key in ('validation', 'test'):
    if split_key in SUMMARY:
        for stat in ('mean', 'std'):
            rows.append(f'{stat},{split_key},' + ','.join(f'{SUMMARY[split_key][k][stat]:.6f}' for k in METRIC_KEYS))
SUMMARY_CSV.write_text('\n'.join(rows) + '\n', encoding='utf-8')

SUMMARY_ZIP = WORKDIR / f'{EXPERIMENT_NAME}_multiseed_summary.zip'
with zipfile.ZipFile(SUMMARY_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [SUMMARY_JSON, SUMMARY_CSV, *(found[s][0] for s in sorted(found))]:
        archive.write(path, path.name)

log_section(f'MULTI-SEED SUMMARY — {len(found)}/{len(SEEDS)} SEED')
missing = sorted(set(SEEDS) - set(found))
if missing:
    print(f'WARNING: seed belum selesai: {missing}')
print(SUMMARY_CSV.read_text(encoding='utf-8'))
from IPython.display import FileLink, display
display(FileLink(SUMMARY_ZIP))
for s in sorted(found):
    seed_zip = seed_paths(s)[3]
    if seed_zip.is_file():
        display(FileLink(seed_zip))


MULTI-SEED SUMMARY — 3/3 SEED
seed,split,precision,recall,map50,map50_95
42,validation,0.850852,0.829049,0.891160,0.605756
42,test,0.897547,0.806689,0.863466,0.578334
123,validation,0.876342,0.833766,0.887909,0.599257
123,test,0.858642,0.813662,0.860551,0.570978
456,validation,0.864115,0.845353,0.896011,0.606290
456,test,0.846682,0.828254,0.853129,0.578740
mean,validation,0.863770,0.836056,0.891693,0.603768
std,validation,0.012748,0.008390,0.004077,0.003916
mean,test,0.867624,0.816202,0.859049,0.576017
std,test,0.026596,0.011005,0.005330,0.004369



/kaggle/working/yolo12s_original_pretrained_ch_rdd2022_multiseed_summary.zip

/kaggle/working/yolo12s_original_pretrained_ch_rdd2022_seed123_results.zip

/kaggle/working/yolo12s_original_pretrained_ch_rdd2022_seed456_results.zip